# 26 užduotis: kaip keičiasi banga?

Programa skaičiuoja bangos reikšmes skirtingose vietose $x$ ir skirtingu laiku $t$. Ji tai daro pagal šią lygtį:

$$u_t = (a^2 + i)\,u_{xx} + \beta\,(|u|^2u)_x + f(x,t), \qquad 0 < x < 1$$

- `NMM` = netiesinis modeliavimas
- Metodas vadinamas `H1`, kraštinės sąlygos `II`. Kraštinės sąlygos = taisyklės dviejuose kraštuose ($x = 0$ ir $x = 1$)
- $u_0 = u_1$ ir $u_N = u_{N-1}$

Mintis: mes žinome tikslų atsakymą, todėl galime patikrinti, kiek mūsų skaičiai klaidingi.

In [69]:
import numpy as np

# Užduoties modelio parametrai.
a = 1.0        # Koeficientas lygtyje.
beta = 1.0     # Netiesinės dalies koeficientas.
T = 1.0        # Iki kurio laiko skaičiuojame.
delta = 1e-8   # Kada kartojamus spėjimus laikome pakankamai panašiais.

## 1. Žinomas atsakymas ir $f$

Čia iš anksto pasirenkame vieną funkciją `u_tikslus`. Ji mums duoda žinomą atsakymą, su kuriuo vėliau palyginsime kompiuterio skaičiavimą.

$f$ nėra išvestinė. Tai papildoma lygties funkcija. Ją apskaičiuojame taip, kad pasirinkta `u_tikslus` tiktų kaip šios lygties atsakymas.

In [70]:
def u_tikslus(x, t):
    # Duodame vietą x ir laiką t, gauname bangos reikšmę.
    return (1 + 1j * t) * np.cos(2 * np.pi * x)


def f(x, t):
    z = 1 + 1j * t
    c = np.cos(2 * np.pi * x)
    s = np.sin(2 * np.pi * x)

    # Išvestinė pagal laiką t.
    u_t = 1j * c
    # Antrosios eilės išvestinė pagal vietą x.
    u_xx = -4 * np.pi ** 2 * z * c
    # Netiesinio reiškinio išvestinė pagal x.
    netiesinis_x = -6 * np.pi * (1 + t ** 2) * z * c ** 2 * s

    # Sudedame jas pagal lygtį ir gauname f reikšmę.
    return u_t - (a ** 2 + 1j) * u_xx - beta * netiesinis_x

## 2. Kaip išsprendžiama lygčių sistema?

Kiekviename laiko žingsnyje reikia rasti bangos reikšmes visuose vietos taškuose. Kiekviena vidinė lygtis susieja tris gretimas reikšmes: ankstesnę, dabartinę ir kitą.

Tuomet tokią sistemą galima išspręsti perkelties metodu.

`perkelties_metodas()` funkcija lygčių sistemą ir grąžina naują visų bangos taškų reikšmių rinkinį.

In [71]:
def thomas_algoritmas(C, F):
    # Iš F ilgio sužinome, kiek yra tarpų. Taškų yra vienu daugiau.
    N = len(F) - 1

    # Čia laikome tarpinius skaičius, kurių reikia sprendžiant sistemą.
    alfa = np.zeros(N + 1, dtype=complex)
    beta_k = np.zeros(N + 1, dtype=complex)
    alfa[1] = 1.0   # Kairio krašto taisyklė: y[0] = y[1].
    beta_k[1] = 0.0

    # Pereiname per lygtis ir paruošiame tarpinius skaičius.
    for j in range(1, N):
        alfa[j + 1] = 1 / (C - alfa[j])
        beta_k[j + 1] = (beta_k[j] + F[j]) / (C - alfa[j])

    # Atgaline tvarka apskaičiuojame ieškomas bangos reikšmes y.
    y = np.zeros(N + 1, dtype=complex)
    y[N] = beta_k[N] / (1 - alfa[N])   # Dešinio krašto taisyklė: y[N] = y[N-1].
    for j in range(N - 1, -1, -1):
        y[j] = alfa[j + 1] * y[j + 1] + beta_k[j + 1]
    return y

## 3. H1 formulė

H1 formulė yra:

$$
\frac{\hat u_j-u_j}{\tau}
=
\frac{k}{2}\left(
\frac{\hat u_{j+1}-2\hat u_j+\hat u_{j-1}}{h^2}
+
\frac{u_{j+1}-2u_j+u_{j-1}}{h^2}
\right)
+\beta\frac{g_{j+1}-g_{j-1}}{2h}
+\frac{f_j+\hat f_j}{2},
\qquad k=a^2+i.
$$

Čia $w_j=(u_j+\hat u_j)/2$ ir $g_j=|w_j|^2w_j$.

Pertvarkius lygtį, naujos reikšmės $\hat u_j$ atsiduria kairėje pusėje. Gauname $y_{j-1}-Cy_j+y_{j+1}=-F_j$, kur $y_j=\hat u_j$:

$$
C=2+\frac{2h^2}{k\tau},
$$

$$
F_j=
(u_{j+1}-2u_j+u_{j-1})
+\frac{2h^2}{k\tau}u_j
+\frac{2h^2}{k}\left(
\beta\frac{g_{j+1}-g_{j-1}}{2h}
+\frac{f_j+\hat f_j}{2}
\right),
\qquad j=1,\ldots,N-1.
$$

In [72]:
def sudaryti_F(u, g, f_vid, h, tau, k, beta):
    F = np.zeros(len(u), dtype=complex)

    u_sena = u[2:] - 2 * u[1:-1] + u[:-2]
    g_x = (g[2:] - g[:-2]) / (2 * h)

    F[1:-1] = (
        u_sena
        + 2 * h ** 2 / (k * tau) * u[1:-1]
        + 2 * h ** 2 / k * (beta * g_x + f_vid[1:-1])
    )
    return F

def sprendimas(N, tau):
    h = 1 / N
    x = np.linspace(0, 1, N + 1)
    k = a ** 2 + 1j
    C = 2 + 2 * h ** 2 / (k * tau)

    # Pradedame nuo žinomo atsakymo ties laiku t = 0.
    u = u_tikslus(x, 0)
    t = 0.0
    max_paklaida = 0.0

    # Išorinis ciklas: apskaičiuojame vis naujesnius laikus iki T.
    while t < T - tau / 2:
        # Seno ir naujo f vidurkis.
        f_vid = (f(x, t) + f(x, t + tau)) / 2       # (f^_j + f_j)/2
        u_senas = u.copy()

        while True:
            # Naudojame seno spėjimo ir dabartinės reikšmės vidurkį.
            u_vid = (u_senas + u) / 2       # (û_j + u_j)/2
            g = np.abs(u_vid) ** 2 * u_vid      # |(û_j + u_j) / 2|^2 * ((û_j + u_j) / 2)

            # Sudarome F, tada perkelties metodas išsprendžia H1 lygčių sistemą.
            F = sudaryti_F(u, g, f_vid, h, tau, k, beta)
            u_naujas = thomas_algoritmas(C, F)
            skirtumas = np.max(np.abs(u_naujas - u_senas))
            if skirtumas < delta:
                break
            u_senas = u_naujas

        u = u_naujas
        t += tau
        max_paklaida = max(max_paklaida, np.max(np.abs(u - u_tikslus(x, t))))

    return u, max_paklaida

## 4. Testas 1: H1 formulės netiktis

Viename $(x,t)$ taške įstatome žinomą sprendinį į H1 formulę ir apskaičiuojame, kiek jos pusės nesutampa. Pradedame nuo $h=\tau=10^{-1}$ ir penkis kartus sumažiname žingsnius dešimt kartų.

Dėl riboto tikslumo apvalinama kiekviena operacija. Kai $h$ ir $\tau$ maži, atimant artimas reikšmes ir dalijant iš $h^2$ ar $\tau$, apvalinimo paklaida gali turėti didesnę įtaką.

In [77]:
def testas1_netiktis():
    # Įstatome žinomą sprendinį į H1 formulę viename x ir t taške.
    print('1 testas. H1 formulės netiktis taške x = 0.39, t = 0.57')
    x, t = 0.39, 0.57
    h = tau = 0.1
    k = a ** 2 + 1j
    ankstesne = None

    # Mažiname h ir tau, bet H1 bangos spėjimų nekartojame.
    for _ in range(5):
        u_m, u_0, u_p = u_tikslus(x - h, t), u_tikslus(x, t), u_tikslus(x + h, t)
        v_m, v_0, v_p = u_tikslus(x - h, t + tau), u_tikslus(x, t + tau), u_tikslus(x + h, t + tau)
        w_m, w_p = (v_m + u_m) / 2, (v_p + u_p) / 2
        g_m, g_p = abs(w_m) ** 2 * w_m, abs(w_p) ** 2 * w_p

        netiktis = abs((v_0 - u_0) / tau
                       - k / 2 * ((v_p - 2 * v_0 + v_m) / h ** 2 + (u_p - 2 * u_0 + u_m) / h ** 2)
                       - beta * (g_p - g_m) / (2 * h)
                       - (f(x, t + tau) + f(x, t)) / 2)
        if ankstesne is None:
            pokytis = ''
        elif netiktis < ankstesne:
            pokytis = f', sumažėjo {ankstesne / netiktis:.3g} karto'
        else:
            pokytis = f', padidėjo {netiktis / ankstesne:.3g} karto'
        print(f'  h = tau = {h:.0e}: netiktis = {netiktis:.3e}{pokytis}')
        ankstesne = netiktis
        h /= 10
        tau /= 10


testas1_netiktis()

1 testas. H1 formulės netiktis taške x = 0.39, t = 0.57
  h = tau = 1e-01: netiktis = 2.766e+00
  h = tau = 1e-02: netiktis = 3.140e-02, sumažėjo 88.1 karto
  h = tau = 1e-03: netiktis = 3.125e-04, sumažėjo 100 karto
  h = tau = 1e-04: netiktis = 3.104e-06, sumažėjo 101 karto
  h = tau = 1e-05: netiktis = 4.301e-06, padidėjo 1.39 karto


## 5. Testas 2: F funkcijos patikra

Pagal žinomą sprendinį sudarome H1 formulės reikšmes ir patikriname, ar `sudaryti_F` grąžina tokį patį $F$. Taip tikriname, ar teisingai užrašyta $F$ funkcija.

In [74]:
# Antras testas tikrina, ar funkcija sudaryti_F teisingai apskaičiuoja F reikšmes.
N_test = 20
h_test = 1 / N_test
tau_test = 0.05
k_test = a ** 2 + 1j
x_test = np.linspace(0, 1, N_test + 1)
t_test = 0.2

u_test = u_tikslus(x_test, t_test)
u_hat_test = u_tikslus(x_test, t_test + tau_test)
w_test = (u_test + u_hat_test) / 2
g_test = np.abs(w_test) ** 2 * w_test
f_vid_test = (f(x_test, t_test) + f(x_test, t_test + tau_test)) / 2

F_test = sudaryti_F(u_test, g_test, f_vid_test, h_test, tau_test, k_test, beta)
F_patikra = np.zeros(N_test + 1, dtype=complex)
for j in range(1, N_test):
    F_patikra[j] = (
        u_test[j + 1] - 2 * u_test[j] + u_test[j - 1]
        + 2 * h_test ** 2 / (k_test * tau_test) * u_test[j]
        + 2 * h_test ** 2 / k_test * (
            beta * (g_test[j + 1] - g_test[j - 1]) / (2 * h_test)
            + f_vid_test[j]
        )
    )

print('2 testas. F funkcijos patikra')
print(f'  didžiausias |F_test - F_patikra| = {np.max(np.abs(F_test - F_patikra)):.3e}')

2 testas. F funkcijos patikra
  didžiausias |F_test - F_patikra| = 2.776e-17


## 6. Testas 3: Thomas algoritmo patikra

Sugeneruojame atsitiktinę kompleksinę bangą $y$, iš jos apskaičiuojame $F$ reikšmes ir paduodame jas Thomas algoritmui. Tada palyginame pradinę bangą su algoritmo atkurta banga.

In [75]:
# Trečias testas tikrina Thomas (perkelties) algoritmą su atsitiktine kompleksine banga.
N_test = 20
C_test = 2.5 + 1j
atsitiktinis = np.random.default_rng(26)
y_tikslus = atsitiktinis.random(N_test + 1) + 1j * atsitiktinis.random(N_test + 1)
y_tikslus[0] = y_tikslus[1]
y_tikslus[N_test] = y_tikslus[N_test - 1]

# Apskaičiuojame F iš sistemos y[j-1] - C*y[j] + y[j+1] = -F[j].
F_test = np.zeros(N_test + 1, dtype=complex)
for j in range(1, N_test):
    F_test[j] = -(y_tikslus[j - 1] - C_test * y_tikslus[j] + y_tikslus[j + 1])

# Thomas algoritmas turi atkurti atsitiktinai sugeneruotą bangą.
y_gautas = thomas_algoritmas(C_test, F_test)
print('3 testas. Thomas algoritmo patikra')
print(f'  didžiausia |y_gautas - y_tikslus| = {np.max(np.abs(y_gautas - y_tikslus)):.3e}')

3 testas. Thomas algoritmo patikra
  didžiausia |y_gautas - y_tikslus| = 3.140e-16


## 7. Testas 4: viso sprendinio paklaida

Sprendžiame bangą per visą laikotarpį nuo $t=0$ iki $T$. Kiekviename laiko žingsnyje ir kiekviename tinklo taške lyginame apskaičiuotą reikšmę su `u_tikslus`; išrenkame didžiausią paklaidą. Pakartojame skaičiavimą keliems tinklo tankumams.

In [76]:
# Ketvirtas testas: viso laikotarpio globali sprendinio paklaida.
print('4 testas. Viso sprendinio paklaida')
N_sarasas = [15, 40, 80, 160]
ankstesne = None
for N_test in N_sarasas:
    tau_testas = 1 / N_test
    _, paklaida = sprendimas(N_test, tau_testas)
    santykis = f', sumažėjo {ankstesne / paklaida:.2f} karto' if ankstesne else ''
    print(f'  h = tau = {1 / N_test:.5f}: didžiausia paklaida = {paklaida:.3e}{santykis}')
    ankstesne = paklaida

4 testas. Viso sprendinio paklaida
  h = tau = 0.06667: didžiausia paklaida = 2.926e+00
  h = tau = 0.02500: didžiausia paklaida = 2.462e+00, sumažėjo 1.19 karto
  h = tau = 0.01250: didžiausia paklaida = 1.117e+00, sumažėjo 2.20 karto
  h = tau = 0.00625: didžiausia paklaida = 7.606e-01, sumažėjo 1.47 karto
